# Attribution Benchmark — Unified Pipeline

> SIF / IF-diag / DVF vs LOO Ground Truth → Spearman + Deletion curve + stat tests.

In [13]:
%run ./functions.ipynb

import time, json, math, os
from functools import lru_cache
from pathlib import Path

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "notebooks").exists() and (p / "requirements.txt").exists():
            return p
    return c

REPO_ROOT = _find_root()
seed_everything(math.ceil(time.time()))
print(f"Device: {DEVICE}")

Device: cuda


## 1. Config & Load Data

In [14]:
# ---------- adjustable ----------
DATASETS = ["pinterest", "yahoo", "ml1m"]   # datasets to benchmark
BENCH_MODELS = ["MLP", "NCF"]
ATTR_METHODS = ["sif", "if_diag", "dvf"]
BENCH_SEEDS = [7, 42, 91]

N_CONTROL = 64
N_VAL = 512
LOO_EPOCHS = 20
LOO_LR = 8e-4
LOO_SEEDS = 5
BASE_EPOCH = 1

# ---------- paths ----------
OUT_DIR = REPO_ROOT / "log" / "notebook_runs"
STEP2_ROOT = OUT_DIR / "step2_training"
BENCH_DIR = OUT_DIR / "attribution_benchmark"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

def _resolve_step2(ds):
    ds = ds.lower()
    cands = sorted([d for d in STEP2_ROOT.iterdir() if d.is_dir() and d.name.endswith(f"_{ds}")],
                   key=lambda d: d.name, reverse=True)
    if cands: return cands[0]
    raise FileNotFoundError(f"No step2 run found for {ds}")

print(f"Datasets: {DATASETS}")
print(f"Models: {BENCH_MODELS}  Methods: {ATTR_METHODS}  Seeds: {BENCH_SEEDS}")

Datasets: ['pinterest', 'yahoo', 'ml1m']
Models: ['MLP', 'NCF']  Methods: ['sif', 'if_diag', 'dvf']  Seeds: [7, 42, 91]


In [15]:
# Load cache + checkpoints — per dataset
def _load_dataset(DS):
    cache = torch.load(REPO_ROOT / "log" / "notebook_cache" / f"prepared_data_{DS}.pt", map_location="cpu", weights_only=False)
    matrix = cache["user_item_matrix"]
    p = PreparedData(
        user_item_matrix=matrix, labels=cache["labels"], sample_ids=cache["sample_ids"],
        user_ids=cache["user_ids"], item_ids=cache["item_ids"],
        num_items=cache["num_items"], num_users=cache["num_users"],
    )
    step2 = _resolve_step2(DS)
    hp = json.load(open(step2 / "hparams.json"))
    return p, step2, hp

def _load_ckpts(step2_dir):
    ac = step2_dir / "all_checkpoints.pt"
    if ac.exists():
        raw = torch.load(ac, map_location="cpu", weights_only=False)
        if isinstance(raw, dict):
            return {k.upper(): v if isinstance(v,list) else [] for k,v in raw.items()}
    ck = {}
    for m in ["MLP","NCF","VAE"]:
        f = step2_dir / f"{m.lower()}_checkpoints"
        if f.exists(): ck[m] = [str(p) for p in sorted(f.glob("epoch*.pt"))]
    return ck

## 2. Helpers

In [16]:
def _sub(p, idxs):
    idxs = torch.tensor(list(idxs), dtype=torch.long)
    return PreparedData(
        user_item_matrix=p.user_item_matrix, labels=p.labels[idxs],
        sample_ids=p.sample_ids[idxs], user_ids=p.user_ids[idxs],
        item_ids=p.item_ids[idxs], num_items=p.num_items, num_users=p.num_users,
    )

def _loader_from(p, bs=16, shuffle=False):
    return DataLoader(RecDataset(p), batch_size=bs, shuffle=shuffle)

def _collect(loader, n):
    ds = loader.dataset
    p = PreparedData(
        user_item_matrix=ds.matrix,
        labels=ds.labels, sample_ids=ds.sids,
        user_ids=ds.user_ids, item_ids=ds.item_ids,
        num_items=ds.matrix.shape[1], num_users=ds.matrix.shape[0],
    )
    idx = torch.randperm(len(p.labels))[:min(n, len(p.labels))]
    return _sub(p, idx)

def _spearman(a, b):
    ra = pd.Series(a).rank(method="average").to_numpy()
    rb = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ra) == 0 or np.std(rb) == 0: return 0.0
    return float(np.corrcoef(ra, rb)[0, 1])

def _stable_seed(idxs, rep, seed, model_name):
    key = sorted([int(x) for x in idxs])
    acc = sum((t+1)*(x+17) for t,x in enumerate(key)) % 2147483647
    name_acc = sum(ord(c) for c in model_name)
    return int((seed*1000003 + acc*1315423911 + rep*2654435761 + name_acc) % 2147483647)

## 3. Main Loop — Multi-Seed LOO + Attribution

In [ ]:
gt_cache_dir = BENCH_DIR / "loo_gt_cache"
gt_cache_dir.mkdir(parents=True, exist_ok=True)

import json as _j

all_rows = []

for DS in [d.lower() for d in DATASETS]:
    prepared, STEP2_DIR, _hp_all = _load_dataset(DS)
    CHECKPOINTS = _load_ckpts(STEP2_DIR)
    NUM_ITEMS = prepared.num_items
    print(f"\n{'#'*60}")
    print(f"# Dataset: {DS}  ({prepared.num_users} users, {NUM_ITEMS} items)")
    print(f"# Step2: {STEP2_DIR}")
    print(f"{'#'*60}")

    for bench_seed in BENCH_SEEDS:
        seed_everything(bench_seed)

        # ---- train/val split ----
        n = prepared.labels.shape[0]
        g = torch.Generator().manual_seed(bench_seed)
        perm = torch.randperm(n, generator=g)
        n_val_samp = int(n * 0.2)

        train_small = _collect(_loader_from(_sub(prepared, perm[n_val_samp:]), 256, False), N_CONTROL)
        val_small   = _collect(_loader_from(_sub(prepared, perm[:n_val_samp]), 256, False), N_VAL)
        train_small.user_item_matrix = train_small.user_item_matrix.to(DEVICE)

        ts_loader = _loader_from(train_small, 16, False)
        vs_loader = _loader_from(val_small, 32, False)

        control_ids = [int(x.item()) for x in train_small.sample_ids]
        ctrl_idx = list(range(len(control_ids)))
        ids_set = set(control_ids)

        for model_name in BENCH_MODELS:
            if model_name not in CHECKPOINTS: continue
            print(f"\n  seed={bench_seed}  model={model_name}")

            cfg = _hp_all.get(model_name, {})
            m = build_model(name=model_name, num_items=NUM_ITEMS, cfg=cfg)
            if BASE_EPOCH is not None:
                ckpt_path = STEP2_DIR / f"{model_name.lower()}_checkpoints" / f"epoch{BASE_EPOCH:02d}.pt"
            else:
                ckpt_path = STEP2_DIR / f"{model_name.lower()}_final_state.pt"
            m.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=True))
            m.to(DEVICE)
            base_state = {k: v.detach().cpu().clone() for k,v in m.state_dict().items()}

            @lru_cache(maxsize=4096)
            def utility(tidxs):
                idxs = list(tidxs)
                if len(idxs) == 0: return 0.5
                sub = _sub(train_small, idxs)
                vals = []
                for rep in range(LOO_SEEDS):
                    s2 = _stable_seed(idxs, rep, bench_seed, model_name)
                    seed_everything(s2)
                    u = build_model(name=model_name, num_items=NUM_ITEMS, cfg=cfg)
                    u.load_state_dict(base_state); u.to(DEVICE)
                    h, _ = train_model(u, _loader_from(sub, 16, False), vs_loader,
                                        epochs=LOO_EPOCHS, lr=LOO_LR, device=DEVICE,
                                        loss_fn="bce", verbose=False,
                                        user_item_matrix=train_small.user_item_matrix,
                                        num_items=NUM_ITEMS)
                    vals.append(float(evaluate_model(u, vs_loader, device=DEVICE)["loss"]))
                return float(np.mean(vals))

            full_u = utility(tuple(sorted(ctrl_idx)))

            gt_file = gt_cache_dir / f"gt_{DS}_{model_name}_seed{bench_seed}_n{N_CONTROL}_ep{BASE_EPOCH}.pt"
            if gt_file.exists():
                gt_data = torch.load(gt_file, map_location="cpu")
                gt = {int(k): float(v) for k,v in gt_data.items() if k != "full_u"}
                print(f"  LOO: cache hit ({len(gt)})")
            else:
                print(f"  LOO: computing {N_CONTROL} samples...")
                gt = {}
                t0 = time.perf_counter()
                for i, sid in enumerate(control_ids):
                    keep = tuple(sorted(j for j in ctrl_idx if j != i))
                    gt[sid] = full_u - utility(keep)
                    if (i+1) % 16 == 0:
                        e = time.perf_counter() - t0
                        print(f"    {i+1}/{N_CONTROL}  ({e:.0f}s, ~{e/(i+1)*(N_CONTROL-i-1):.0f}s left)")
                torch.save({"full_u": full_u, **{str(k):v for k,v in gt.items()}}, gt_file)

            gt_vec = [gt.get(sid, 0.0) for sid in control_ids]

            sif_scores = {}
            if "sif" in ATTR_METHODS:
                t0 = time.perf_counter()
                sif_scores = compute_sif(m, vs_loader, ts_loader, max_samples=N_CONTROL, device=DEVICE)
                v = [sif_scores.get(sid, 0.0) for sid in control_ids]
                all_rows.append({"dataset": DS, "seed": bench_seed, "model": model_name, "method": "SIF",
                                "spearman": _spearman(gt_vec, v),
                                "time": round(time.perf_counter()-t0, 1)})
                print(f"  SIF        ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

            ifd_scores = {}
            if "if_diag" in ATTR_METHODS:
                t0 = time.perf_counter()
                ifd_scores = compute_if_diag(m, vs_loader, ts_loader, ids_set, device=DEVICE)
                v = [ifd_scores.get(sid, 0.0) for sid in control_ids]
                all_rows.append({"dataset": DS, "seed": bench_seed, "model": model_name, "method": "IF-diag",
                                "spearman": _spearman(gt_vec, v),
                                "time": round(time.perf_counter()-t0, 1)})
                print(f"  IF-diag    ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

            dvf_scores = {}
            if "dvf" in ATTR_METHODS and CHECKPOINTS.get(model_name, []):
                t0 = time.perf_counter()
                def _builder():
                    return build_model(name=model_name, num_items=NUM_ITEMS, cfg=cfg)
                dvf_total, _, _ = compute_dvf_stage(_builder, CHECKPOINTS[model_name],
                                                     ts_loader, vs_loader,
                                                     max_samples=N_CONTROL, device=DEVICE)
                dvf_scores = dvf_total
                v = [dvf_scores.get(sid, 0.0) for sid in control_ids]
                all_rows.append({"dataset": DS, "seed": bench_seed, "model": model_name, "method": "DVF",
                                "spearman": _spearman(gt_vec, v),
                                "time": round(time.perf_counter()-t0, 1)})
                print(f"  DVF        ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

            rv, rng = [], np.random.default_rng(bench_seed)
            for _ in range(10):
                rs = {sid: float(len(control_ids)-i) for i,sid in enumerate(rng.permutation(control_ids))}
                rv.append(_spearman(gt_vec, [rs.get(sid,0.0) for sid in control_ids]))
            all_rows.append({"dataset": DS, "seed": bench_seed, "model": model_name, "method": "Random",
                            "spearman": float(np.median(rv)), "time": 0.0})
            print(f"  Random     ρ={all_rows[-1]['spearman']:+.4f}")

print(f"\nDone — {len(all_rows)} total rows")


############################################################
# Dataset: pinterest  (19154 users, 8761 items)
# Step2: H:\RecAcc\log\notebook_runs\step2_training\20260728_093613_pinterest
############################################################

  seed=7  model=MLP
  LOO: computing 64 samples...
    16/64  (74s, ~222s left)


## 4. Summary & Tiers

In [ ]:
detail_df = pd.DataFrame(all_rows)
detail_df["abs_spearman"] = detail_df["spearman"].abs()

save_table(detail_df, str(BENCH_DIR / "attribution_detail.csv"))

# ---- One CSV per dataset ----
for ds in sorted(detail_df["dataset"].unique()):
    sub = detail_df[detail_df["dataset"] == ds]
    agg_ds = sub.groupby(["model", "method"], as_index=False).agg(
        spearman_mean=("spearman", "mean"),
        spearman_std=("spearman", "std"),
        abs_mean=("abs_spearman", "mean"),
        abs_std=("abs_spearman", "std"),
        time_mean=("time", "mean"),
        n_runs=("seed", "count"),
    ).sort_values(["model", "abs_mean"], ascending=[True, False])
    save_table(agg_ds, str(BENCH_DIR / f"attribution_summary_{ds}.csv"))
    print(f"\n--- {ds} ---")
    for _, r in agg_ds.iterrows():
        m, s = r['abs_mean'], r['abs_std']
        tier = 'STRONG' if m>0.15 and s<0.1 else 'MODERATE' if m>0.10 else 'WEAK' if m>0.03 else 'INVALID'
        print(f"  {r['model']:5s} {r['method']:10s}  |ρ|={m:+.4f}±{s:.4f}  raw={r['spearman_mean']:+.4f}  {tier}")

# ---- Cross-dataset summary ----
cross = detail_df.groupby(["model", "method"], as_index=False).agg(
    spearman_mean=("spearman", "mean"),
    spearman_std=("spearman", "std"),
    abs_mean=("abs_spearman", "mean"),
    abs_std=("abs_spearman", "std"),
    time_mean=("time", "mean"),
    n_runs=("seed", "count"),
    n_datasets=("dataset", "nunique"),
).sort_values(["model", "abs_mean"], ascending=[True, False])
save_table(cross, str(BENCH_DIR / "attribution_summary_cross.csv"))
display(agg_ds)


print(f"\n--- Cross-dataset ---")
for _, r in cross.iterrows():
    m, s = r['abs_mean'], r['abs_std']
    tier = 'STRONG' if m>0.15 and s<0.1 else 'MODERATE' if m>0.10 else 'WEAK' if m>0.03 else 'INVALID'
    print(f"  {r['model']:5s} {r['method']:10s}  |ρ|={m:+.4f}±{s:.4f}  n={r['n_runs']}  ds={r['n_datasets']}  {tier}")

print(f"\nSaved to: {BENCH_DIR}")

NameError: name 'all_rows' is not defined